# LeafRust — обучение / дообучение MobileNetV3 → TFLite (remote)

Удалённый Jupyter + NVIDIA GPU. Датасет / веса / baseline TFLite **скачиваются сами**, если их нет.

### HF token (рядом со скриптами)

```bash
cp scripts/huggingface.token.example scripts/huggingface.token
# вставьте токен с https://huggingface.co/settings/tokens
```

Также читаются: `scripts/.hf_token`, `scripts/hf_token.txt`, `.secrets/huggingface.token`, env `HF_TOKEN`.

### На сервере один раз

```bash
git clone https://github.com/reinethernal/leafrust.git && cd leafrust
python3 -m venv .venv-gpu && source .venv-gpu/bin/activate
pip install -U pip
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
pip install -r scripts/requirements-train-gpu.txt jupyter ipykernel matplotlib
cp scripts/huggingface.token.example scripts/huggingface.token   # + вставить токен
python -m ipykernel install --user --name leafrust-gpu --display-name "LeafRust GPU"
jupyter lab --ip 0.0.0.0 --port 8888 --no-browser
```

Откройте `scripts/train_mobilenet.ipynb`.  
Если репозитория рядом нет — **ячейка 1 сама сделает** `git clone` в `~/leafrust` (или в `LEAFRUST_ROOT`).


## 1. Конфиг

In [ ]:
from pathlib import Path
import os
import platform

def find_repo() -> Path:
    env = os.environ.get("LEAFRUST_ROOT")
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "scripts" / "train_mobilenet_torch.py").exists():
            return p
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "scripts" / "train_mobilenet_torch.py").exists():
            return cand
    raise FileNotFoundError(
        "Не найден корень LeafRust. cd в репозиторий или задайте LEAFRUST_ROOT."
    )

REPO = find_repo()
SCRIPTS = REPO / "scripts"

DATA_DIR = Path(os.environ.get("LEAFRUST_DATA", REPO / "data" / "plantvillage")).expanduser()
MAX_PER_CLASS = None  # 50 = smoke-test; None = полный датасет

EXPORT_DIR = Path(os.environ.get("LEAFRUST_EXPORT", REPO / "data" / "exports")).expanduser()
OUT_TFLITE = EXPORT_DIR / "plantvillage_mobilenet.tflite"
CHECKPOINT_DIR = REPO / "data" / "checkpoints"
BEST_CKPT = CHECKPOINT_DIR / "best_torch.pt"

WRITE_TO_ANDROID_ASSETS = False
if WRITE_TO_ANDROID_ASSETS:
    OUT_TFLITE = (
        REPO / "android" / "app" / "src" / "main" / "assets" / "models" / "plantvillage_mobilenet.tflite"
    )

BATCH = int(os.environ.get("LEAFRUST_BATCH", "32"))
WORKERS = int(os.environ.get("LEAFRUST_WORKERS", "4" if platform.system() != "Windows" else "0"))
IMAGE_SIZE = 224
HEAD_EPOCHS = 3
FT_EPOCHS = 5
UNFREEZE_BLOCKS = 6
LR_HEAD = 1e-3
LR_FT = 1e-4
VAL_SPLIT = 0.15
SEED = 42
USE_AMP = True
SKIP_TFLITE = False

RESUME = False  # True → дообучение с BEST_CKPT
DOWNLOAD_BASELINE_MODELS = True  # скачать текущий TFLite с CDN, если нет локально

print("host:", platform.node(), platform.system())
print("REPO:", REPO)
print("DATA:", DATA_DIR)
print("OUT:", OUT_TFLITE)
print("RESUME:", RESUME, "| ckpt:", BEST_CKPT.exists())


## 2. GPU, импорты, HF token

In [ ]:
import sys
sys.path.insert(0, str(SCRIPTS))

import json
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from IPython.display import display, clear_output

from hf_auth import describe_token_source, load_hf_token, login_hf
from train_mobilenet_torch import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    SoftmaxWrapper,
    build_model,
    ensure_background_images,
    export_onnx,
    onnx_to_tflite,
    require_cuda,
    run_epoch,
    set_backbone_trainable,
)

tok = login_hf(load_hf_token())
print("HF token source:", describe_token_source(), "| present:", bool(tok))
if not tok:
    print(
        "WARNING: нет токена. Скопируйте scripts/huggingface.token.example → "
        "scripts/huggingface.token и вставьте HF token."
    )

device = require_cuda()
print("cuda devices:", torch.cuda.device_count())


## 3. Автоскачивание датасета и моделей

Если нет ImageFolder / ImageNet-весов / baseline TFLite — качает автоматически.


In [ ]:
from ensure_train_assets import (
    ensure_baseline_models,
    ensure_plantvillage,
    ensure_torchvision_weights,
)

ensure_plantvillage(REPO, DATA_DIR, max_per_class=MAX_PER_CLASS)
ensure_torchvision_weights()

if DOWNLOAD_BASELINE_MODELS:
    baselines = ensure_baseline_models(EXPORT_DIR)
    print("baseline models:", {k: str(v) for k, v in baselines.items()})

# Для RESUME: если нет best_torch.pt — обучение с ImageNet; TFLite baseline только для справки/сравнения
print("dataset OK, torchvision weights OK")


## 4. DataLoader

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ensure_background_images(DATA_DIR)

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.1, 0.1, 0.1, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

full = datasets.ImageFolder(str(DATA_DIR), transform=train_tf)
full_val = datasets.ImageFolder(str(DATA_DIR), transform=val_tf)
class_names = list(full.classes)

indices = list(range(len(full)))
random.shuffle(indices)
n_val = max(1, int(len(full) * VAL_SPLIT))
val_idx, train_idx = indices[:n_val], indices[n_val:]

train_loader = DataLoader(
    Subset(full, train_idx),
    batch_size=BATCH,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=True,
    persistent_workers=WORKERS > 0,
)
val_loader = DataLoader(
    Subset(full_val, val_idx),
    batch_size=BATCH,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=True,
    persistent_workers=WORKERS > 0,
)

print(f"classes={len(class_names)} train={len(train_idx)} val={len(val_idx)} batch={BATCH} workers={WORKERS}")


In [ ]:
xb, yb = next(iter(train_loader))
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
imgs = (xb[:8].cpu() * std + mean).clamp(0, 1).permute(0, 2, 3, 1).numpy()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img, yi in zip(axes.flat, imgs, yb[:8].tolist()):
    ax.imshow(img)
    ax.set_title(class_names[yi][:28], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Обучение

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
best_path = BEST_CKPT
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP) if USE_AMP else None

model = build_model(len(class_names), pretrained=not RESUME).to(device)
if RESUME:
    if not best_path.exists():
        raise FileNotFoundError(f"RESUME=True, нет {best_path}")
    ckpt0 = torch.load(best_path, map_location=device, weights_only=False)
    if ckpt0.get("classes") and list(ckpt0["classes"]) != class_names:
        print("WARNING: классы в ckpt отличаются от датасета")
    model.load_state_dict(ckpt0["model"], strict=False)
    print(f"resume {best_path} val_acc={ckpt0.get('val_acc')}")

history = {"tag": [], "epoch": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}


def plot_history():
    clear_output(wait=True)
    if not history["epoch"]:
        return
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
    xs = range(1, len(history["epoch"]) + 1)
    ax1.plot(xs, history["train_loss"], label="train")
    ax1.plot(xs, history["val_loss"], label="val")
    ax1.set_title("Loss")
    ax1.legend()
    ax2.plot(xs, history["train_acc"], label="train")
    ax2.plot(xs, history["val_acc"], label="val")
    ax2.set_title("Accuracy")
    ax2.legend()
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    i = len(history["epoch"]) - 1
    print(
        f"{history['tag'][i]} {history['epoch'][i]}  "
        f"train loss={history['train_loss'][i]:.4f} acc={history['train_acc'][i]:.3f}  "
        f"val loss={history['val_loss'][i]:.4f} acc={history['val_acc'][i]:.3f}"
    )


def train_phase(epochs: int, lr: float, train_backbone: bool, tag: str) -> float:
    set_backbone_trainable(model, train_backbone, UNFREEZE_BLOCKS if train_backbone else 0)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    best_acc = -1.0
    print(f"=== {tag}: epochs={epochs} lr={lr} trainable={sum(p.numel() for p in params):,} ===")
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, device, opt, scaler)
        va_loss, va_acc = run_epoch(model, val_loader, device, None, None)
        history["tag"].append(tag)
        history["epoch"].append(f"{tag} {epoch}/{epochs}")
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        plot_history()
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(
                {"model": model.state_dict(), "classes": class_names, "val_acc": va_acc},
                best_path,
            )
            print(f"  saved {best_path} (val_acc={va_acc:.3f})")
    return best_acc

print(type(model).__name__, "on", device)


In [ ]:
if HEAD_EPOCHS > 0:
    train_phase(HEAD_EPOCHS, LR_HEAD, train_backbone=False, tag="head")

if FT_EPOCHS > 0:
    train_phase(FT_EPOCHS, LR_FT, train_backbone=True, tag="finetune")

print("best val_acc:", max(history["val_acc"]) if history["val_acc"] else None)


## 6. Экспорт TFLite

In [ ]:
ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()
print("loaded val_acc=", ckpt.get("val_acc"))

OUT_TFLITE.parent.mkdir(parents=True, exist_ok=True)
# Сохраним предыдущий baseline, если перезаписываем
prev = OUT_TFLITE
if prev.exists():
    bak = prev.with_suffix(".tflite.bak")
    shutil.copy2(prev, bak)
    print("backup previous:", bak)

labels_txt = OUT_TFLITE.with_name("labels.txt")
labels_txt.write_text("\n".join(class_names) + "\n", encoding="utf-8")
OUT_TFLITE.with_suffix(".labels.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2), encoding="utf-8"
)

onnx_path = CHECKPOINT_DIR / "leafrust_mobilenet_v3.onnx"
export_onnx(SoftmaxWrapper(model), onnx_path, IMAGE_SIZE)

if SKIP_TFLITE:
    print("SKIP_TFLITE — onnx:", onnx_path)
else:
    onnx_to_tflite(onnx_path, OUT_TFLITE)
    print("TFLite:", OUT_TFLITE, f"({OUT_TFLITE.stat().st_size / 1e6:.2f} MB)")


## 7. Скачать на локальный ПК

```bash
scp user@HOST:~/leafrust/data/exports/plantvillage_mobilenet.tflite \
    ./android/app/src/main/assets/models/
scp user@HOST:~/leafrust/data/exports/labels.txt \
    ./android/app/src/main/assets/models/
```

Или через Jupyter file browser: `data/exports/`.
